In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName("MySparkApp2")\
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/07 10:14:04 WARN Utils: Your hostname, MacBook-Pro-6.local, resolves to a loopback address: 127.0.0.1; using 100.100.165.223 instead (on interface en0)
26/09/07 10:14:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 10:14:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
listings = spark.read.csv("/Users/kamari/Documents/project_info/air_bnb/listings.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",", 
    quote='"',
    escape='"', 
    multiLine=True,
    mode="PERMISSIVE" 
)


In [3]:
reviews = spark.read.csv("/Users/kamari/Documents/project_info/air_bnb/reviews.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",", 
    quote='"',
    escape='"', 
    multiLine=True,
    mode="PERMISSIVE" 
)


In [21]:
# 1. Count the number of reviews per listing using the "reviews" dataset

reviews\
.groupBy(reviews.listing_id)\
.count()\
.show(10)

+-------------------+-----+
|         listing_id|count|
+-------------------+-----+
|           26139592|  173|
|           54369238|   87|
| 756752556396798800|   22|
|1147092019793504034|  102|
|1604760798212891657|   11|
|           44180048|  401|
|           45024912|  134|
| 787023821243442039|  150|
| 904392960639597224|   23|
|1625791020781487478|    5|
+-------------------+-----+
only showing top 10 rows


+-------------------------------------+
|count(calculated_host_listings_count)|
+-------------------------------------+
|                                  490|
+-------------------------------------+



In [45]:
# 2. Compute the total number of listings and average review score per host
import pyspark.sql.functions as f

listings\
.groupBy(listings.host_name)\
.agg(f.count("id"),f.avg("review_scores_rating"))\
.show()

+---------+---------+-------------------------+
|host_name|count(id)|avg(review_scores_rating)|
+---------+---------+-------------------------+
|   Tanjin|        1|                     4.93|
|    Tyler|        1|                     4.67|
|     Chad|        5|                    4.758|
| Samantha|        1|                      5.0|
| Abdullah|        2|                    4.865|
|    Scott|       21|        4.948333333333333|
|   Kristy|        1|                     4.86|
|     Rich|        1|                      5.0|
|    Farah|        1|                     4.86|
|    Grace|        2|                     4.86|
|    Terra|        6|                    4.575|
| Vertesol|        1|                      4.5|
| Cathleen|        2|                     NULL|
| Mohammed|        1|                      3.7|
|  Adriana|        6|                    4.125|
|    James|        2|                    4.945|
|  Flerida|        4|                   4.6725|
|     Umit|        1|                   

+----------------------+
|neighbourhood_cleansed|
+----------------------+
|            THIRD WARD|
|            SIXTH WARD|
|           SECOND WARD|
|            SIXTH WARD|
|           SECOND WARD|
|       FOURTEENTH WARD|
|        FIFTEENTH WARD|
|         ELEVENTH WARD|
|            SIXTH WARD|
|        FIFTEENTH WARD|
|            TENTH WARD|
|        FIFTEENTH WARD|
|            NINTH WARD|
|       FOURTEENTH WARD|
|            FIFTH WARD|
|            TENTH WARD|
|        FIFTEENTH WARD|
|            NINTH WARD|
|       FOURTEENTH WARD|
|        FIFTEENTH WARD|
+----------------------+
only showing top 20 rows


In [61]:
#2.1  Reviews and Average Rating by Neighborhood For each neighborhood, calculate: the total number of listings the average review score

listings\
.groupBy(listings.neighbourhood_cleansed)\
.agg(f.count('id'),f.avg("review_scores_rating"))\
.show()

+----------------------+---------+-------------------------+
|neighbourhood_cleansed|count(id)|avg(review_scores_rating)|
+----------------------+---------+-------------------------+
|            FIRST WARD|       12|        4.623636363636364|
|            FIFTH WARD|       14|        4.310833333333334|
|           SECOND WARD|       41|       4.7195121951219505|
|           EIGHTH WARD|       13|                    4.949|
|       THIRTEENTH WARD|       41|        4.816315789473684|
|            TENTH WARD|       56|        4.742727272727272|
|       FOURTEENTH WARD|       32|        4.931379310344829|
|            NINTH WARD|       42|        4.802564102564102|
|         ELEVENTH WARD|       32|       3.9843333333333333|
|            THIRD WARD|       43|        4.729655172413794|
|            SIXTH WARD|      117|        4.741485148514854|
|        FIFTEENTH WARD|       14|        4.923571428571429|
|          SEVENTH WARD|       14|        4.700714285714286|
|          TWELFTH WARD|

In [73]:
#2.2 — Host Review Activity. For each host, calculate: the total number of reviews across all of their listings the average number of reviews per listing
from pyspark.sql import functions as f

listings\
.groupBy(listings.host_name)\
.agg((f.sum(listings.number_of_reviews)), f.avg(listings.number_of_reviews))\
.show()


+---------+----------------------+----------------------+
|host_name|sum(number_of_reviews)|avg(number_of_reviews)|
+---------+----------------------+----------------------+
|   Tanjin|                    28|                  28.0|
|    Tyler|                    54|                  54.0|
|     Chad|                   224|                  44.8|
| Samantha|                     3|                   3.0|
| Abdullah|                   363|                 181.5|
|    Scott|                   574|    27.333333333333332|
|   Kristy|                    14|                  14.0|
|     Rich|                     5|                   5.0|
|    Farah|                   173|                 173.0|
|    Grace|                    43|                  21.5|
|    Terra|                    53|     8.833333333333334|
| Vertesol|                    24|                  24.0|
| Cathleen|                     0|                   0.0|
| Mohammed|                    10|                  10.0|
|  Adriana|   

In [79]:
#2.3 — Room Type Summary. For each room type, calculate: the total number of listings, the average yearly availability

listings\
.groupBy(listings.room_type)\
.agg(f.count(listings.id),f.avg(listings.availability_365))\
.show()
    

+---------------+---------+---------------------+
|      room_type|count(id)|avg(availability_365)|
+---------------+---------+---------------------+
|    Shared room|        1|                141.0|
|     Hotel room|        9|    288.1111111111111|
|Entire home/apt|      345|    252.8608695652174|
|   Private room|      135|   215.16296296296295|
+---------------+---------+---------------------+



In [ ]:
listings\
.select(listings.name,listings.number_of_reviews)\
.sort(f.desc("number_of_reviews"))\
.limit(10)\
.show()

In [86]:
reviews.show()

+----------+------------------+----------+-----------+---------------+--------------------+
|listing_id|                id|      date|reviewer_id|  reviewer_name|            comments|
+----------+------------------+----------+-----------+---------------+--------------------+
|   2992450|          15066586|2014-07-01|   16827297|        Kristen|Large apartment; ...|
|   2992450|          21810844|2014-10-24|   22648856|    Christopher|This may be a lit...|
|   2992450|          27434334|2015-03-04|      45406|          Altay|The apartment was...|
|   2992450|          28524578|2015-03-25|    5485362|           John|Kenneth was ready...|
|   2992450|          35913434|2015-06-23|   15772025|       Jennifer|We were pleased t...|
|   2992450|          38893053|2015-07-19|   11614467|      Stephanie|The flat is not i...|
|   2992450|          57989144|2015-12-31|   28580637|          Betty|The apartment was...|
|   2992450|457366954464901293|2021-09-22|  413779309|       Carolina|The place 

In [97]:
# 3: Find the top ten listings with the highest number of reviews
from pyspark.sql import functions as f

reviews\
.groupBy("listing_id")\
.count()\
.orderBy("count",ascending = False)\
.limit(10)\
.show()

listings\
.select(listings.name,listings.number_of_reviews)\
.sort(f.desc("number_of_reviews"))\
.limit(10)\
.show()

+----------+-----+
|listing_id|count|
+----------+-----+
|  25549565|  996|
|  10768745|  906|
|  28722270|  749|
|  28868857|  692|
|  16531782|  604|
|  33558235|  505|
|  38321579|  490|
|  32993402|  488|
|   9501054|  469|
|   5651579|  406|
+----------+-----+

+--------------------+-----------------+
|                name|number_of_reviews|
+--------------------+-----------------+
|Quiet and Pretty ...|              996|
|Alb hospital area...|              906|
|Historic Loft Sui...|              749|
|Cozy Garden 2-Bed...|              692|
|On a little park ...|              604|
|Historic, Spaciou...|              505|
|     Comfy and quiet|              490|
|Historic Full Ame...|              488|
|Spacious suite wi...|              469|
|Large studio apt ...|              406|
+--------------------+-----------------+



In [118]:
# 4. Find the top five neighborhoods with the most listings

listings\
.groupBy(listings.neighbourhood_cleansed)\
.count()\
.orderBy("count", ascending = False)\
.limit(5)\
.show()

listings\
.groupBy(listings.neighbourhood_cleansed)\
.agg(f.count("neighbourhood_cleansed").alias('neighbourhood_count'))\
.sort(f.desc("neighbourhood_count"))\
.limit(5)\
.show()

+----------------------+-----+
|neighbourhood_cleansed|count|
+----------------------+-----+
|            SIXTH WARD|  117|
|            TENTH WARD|   56|
|            THIRD WARD|   43|
|            NINTH WARD|   42|
|           SECOND WARD|   41|
+----------------------+-----+

+----------------------+-------------------+
|neighbourhood_cleansed|neighbourhood_count|
+----------------------+-------------------+
|            SIXTH WARD|                117|
|            TENTH WARD|                 56|
|            THIRD WARD|                 43|
|            NINTH WARD|                 42|
|           SECOND WARD|                 41|
+----------------------+-------------------+



In [22]:
# 5. Get a data frame with the following four columns:
# * Listing's ID
# * Listing's name
# * Reviewer's name
# * Review's comment
# Use "join" to combine data from two datasets

new_df_four_col = listings\
.join(reviews, listings.id == reviews.listing_id, "outer")\
.select(listings.id, listings.name, reviews.reviewer_name, reviews.comments)
    
new_df_four_col.show()

+--------+--------------------+--------------+--------------------+
|      id|                name| reviewer_name|            comments|
+--------+--------------------+--------------+--------------------+
|13083497|Pristine Cape Cod...|           Pam|A very clean spac...|
|13083497|Pristine Cape Cod...|           Pam|We were happy to ...|
|13083497|Pristine Cape Cod...|       Sabarni|Akhilesh helped h...|
|13083497|Pristine Cape Cod...|         Renee| Had a lovely sta...|
|13083497|Pristine Cape Cod...|     Elizabeth|I rented Chris' h...|
|13083497|Pristine Cape Cod...|         Wendy|The house itself ...|
|13083497|Pristine Cape Cod...|       Carolee|We had a lovely s...|
|13083497|Pristine Cape Cod...|           Lee|This was a great ...|
|13083497|Pristine Cape Cod...|Tareq & Halima|If you plan to vi...|
|13083497|Pristine Cape Cod...|           Ana|It was a great ex...|
|13083497|Pristine Cape Cod...|           Pat|Lovely older home...|
|13083497|Pristine Cape Cod...|        Donald|I 

In [30]:
reviews.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: long (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)



In [4]:
# 6.Get top five listings with the highest average review comment length. Only return listings with at least 5 reviews
# Use the "length" function from the "pyspark.sql.functions" to get a lenght of a review
from pyspark.sql import functions as f

top_five_avg_review = listings\
.join(reviews, listings.id == reviews.listing_id, "inner")\
.select(listings.name, f.length(reviews.comments).alias('comment_length'))\
.groupBy(listings.name)\
.agg(f.avg('comment_length').alias('avg_review_comment_len'),f.count('*').alias('review_count'))\
.filter(f.col('review_count') >= 5)\
.sort(f.desc('avg_review_comment_len'))\
.limit(5)


top_five_avg_review.show()


+--------------------+----------------------+------------+
|                name|avg_review_comment_len|review_count|
+--------------------+----------------------+------------+
|Luxury 2 bedroom ...|     604.7777777777778|           9|
|Large room with w...|                 504.8|           5|
|Historic Brownsto...|     496.1666666666667|           6|
|Historic Washingt...|     426.8333333333333|          12|
|Center Sq 2BR Bro...|     386.3103448275862|          29|
+--------------------+----------------------+------------+



In [6]:
reviews.show()

+----------+------------------+----------+-----------+---------------+--------------------+
|listing_id|                id|      date|reviewer_id|  reviewer_name|            comments|
+----------+------------------+----------+-----------+---------------+--------------------+
|   2992450|          15066586|2014-07-01|   16827297|        Kristen|Large apartment; ...|
|   2992450|          21810844|2014-10-24|   22648856|    Christopher|This may be a lit...|
|   2992450|          27434334|2015-03-04|      45406|          Altay|The apartment was...|
|   2992450|          28524578|2015-03-25|    5485362|           John|Kenneth was ready...|
|   2992450|          35913434|2015-06-23|   15772025|       Jennifer|We were pleased t...|
|   2992450|          38893053|2015-07-19|   11614467|      Stephanie|The flat is not i...|
|   2992450|          57989144|2015-12-31|   28580637|          Betty|The apartment was...|
|   2992450|457366954464901293|2021-09-22|  413779309|       Carolina|The place 

In [ ]:
# 6.5. Using the "join" operator find listings without reviews.


no_reviews = listings\
.join(reviews, listings.id == reviews.listing_id, 'left_anti')\


no_reviews.select('name').show()

+--------------------+
|                name|
+--------------------+
|               Homey|
|Clean lines and a...|
|       Room for rent|
|Whalen 3 at The A...|
|Whalen 4 at The A...|
|Knickerbocker 1 T...|
|Knickerbocker 3 T...|
|纽约州首府的温馨小屋|
|Downtown Brownsto...|
|Cozy Bedrooms in ...|
|"The LEO Home" Pr...|
|   Funky loft studio|
|"The Albany Allen...|
|Albany Vacation R...|
|"The LEO Home" Pr...|
|"The Albany Allen...|
|          Homey Room|
|staycation CAP re...|
|"The LEO Home" Pr...|
|Short term across...|
+--------------------+
only showing top 20 rows


In [ ]:
#7 — Most Active Reviewers: Find the top 5 reviewers who have written the most reviews.

reviews\
.groupBy('reviewer_id', 'reviewer_name')\
.agg(f.count("reviewer_id").alias("num_reviews"))\
.sort(f.desc("num_reviews"))\
.limit(5)\
.show()


+-------------+-----------+
|reviewer_name|num_reviews|
+-------------+-----------+
|      Michael|        304|
|        David|        226|
|       Daniel|        210|
|         John|        203|
|        Sarah|        171|
+-------------+-----------+



In [31]:
#8 — Neighborhood Review Summary: Only keep neighborhoods with at least 10 reviews, then sort the results by total number of reviews from highest to lowest.
from pyspark.sql import functions as f

ten_reviews = listings.join(reviews, listings.id == reviews.listing_id, "inner")

ten_reviews\
.groupBy(listings.neighbourhood_cleansed)\
.agg(f.count(reviews.reviewer_id).alias('review_count'))\
.filter(f.col('review_count' )>= 10)\
.sort(f.desc('review_count'))\
.show()

+----------------------+------------+
|neighbourhood_cleansed|review_count|
+----------------------+------------+
|            SIXTH WARD|        8197|
|           SECOND WARD|        5615|
|       FOURTEENTH WARD|        2913|
|            NINTH WARD|        1933|
|       THIRTEENTH WARD|        1765|
|            TENTH WARD|        1527|
|            THIRD WARD|        1354|
|        FIFTEENTH WARD|        1316|
|           EIGHTH WARD|        1059|
|           FOURTH WARD|         991|
|         ELEVENTH WARD|         976|
|          SEVENTH WARD|         656|
|            FIRST WARD|         444|
|            FIFTH WARD|         277|
|          TWELFTH WARD|          23|
+----------------------+------------+



In [24]:
#9 — Hosts With Long Reviews: Find hosts whose listings have an average review comment length greater than 100 characters.

from pyspark.sql import functions as f

long_reviews = listings.join(reviews, listings.id == reviews.listing_id, 'inner')

long_reviews\
.select(listings.host_name, f.length(reviews.comments).alias('reviews_comments_len'))\
.groupBy(listings.host_name)\
.agg(f.avg(f.col('reviews_comments_len')).alias('avg_review_len'))\
.filter(f.col('avg_review_len') > 100 )\
.show()


+---------+------------------+
|host_name|    avg_review_len|
+---------+------------------+
|   Tanjin|245.71428571428572|
|    Tyler| 232.7962962962963|
|     Chad|197.86607142857142|
| Samantha|             201.0|
| Abdullah| 203.7382920110193|
|    Scott| 179.3623693379791|
|   Kristy|263.35714285714283|
|     Rich|             292.4|
|    Farah|185.27167630057804|
|    Grace|160.13953488372093|
|    Terra| 258.6792452830189|
| Vertesol|166.41666666666666|
| Mohammed|             200.7|
|  Adriana| 411.8333333333333|
|    James|231.95530726256985|
|  Flerida|272.26771653543307|
|     Umit|             575.5|
|     Liza|233.60526315789474|
| AnnMarie|205.41666666666666|
|    Jason| 200.1147132169576|
+---------+------------------+
only showing top 20 rows


26/09/05 18:28:04 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:707)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1552)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:484)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$s

In [ ]:
#10 — Neighborhoods With Highly Rated Listings: Find neighborhoods whose listings have an average review score greater than 4.7.
from pyspark.sql import functions as f

listings\
.groupBy(listings.neighbourhood_cleansed)\
.agg(f.avg(listings.review_scores_rating).alias('avg_review_rating'))\
.filter(f.col('avg_review_rating')>4.7)\
.show()

+----------------------+------------------+
|neighbourhood_cleansed| avg_review_rating|
+----------------------+------------------+
|           SECOND WARD|4.7195121951219505|
|           EIGHTH WARD|             4.949|
|       THIRTEENTH WARD| 4.816315789473684|
|            TENTH WARD| 4.742727272727272|
|       FOURTEENTH WARD| 4.931379310344829|
|            NINTH WARD| 4.802564102564102|
|            THIRD WARD| 4.729655172413794|
|            SIXTH WARD| 4.741485148514854|
|        FIFTEENTH WARD| 4.923571428571429|
|          SEVENTH WARD| 4.700714285714286|
|          TWELFTH WARD|              4.84|
|           FOURTH WARD| 4.777333333333333|
+----------------------+------------------+



In [27]:
#11 — Listings With Many Unique Reviewers: Find listings that have received reviews from more than 20 unique reviewers.

unique_reviews = listings.join(reviews, listings.id == reviews.listing_id, 'inner')

unique_reviews\
.groupBy(listings.name)\
.agg(f.count_distinct(reviews.reviewer_id).alias('reviews_count'))\
.filter(f.col('reviews_count')>20)\
.show()


+--------------------+-------------+
|                name|reviews_count|
+--------------------+-------------+
|Chic Madison Park...|           42|
|Historic Queen St...|          374|
|Washington Parksi...|          271|
|The Down Under Suite|           85|
|Garden level apar...|           42|
|Quiet cozy apartm...|           84|
|Quiet and Pretty ...|          936|
|WOW ★ Bright Apt ...|          129|
|      The Comfort II|           31|
|The Loft Suite @ ...|          189|
|Private Room in t...|           22|
|Downtown Albany 2...|          246|
|$53($25 foreign s...|           89|
|Studio in Heart o...|          188|
|Modern Cottage-Pe...|          149|
|Renovated, privat...|          136|
|     Comfy and quiet|          288|
|*Beautiful* Spaci...|           30|
|Chic-romantic, ce...|          268|
|ParkSouth Brickho...|           29|
+--------------------+-------------+
only showing top 20 rows


In [ ]:
#13 — Neighborhood Review Activity: Find neighborhoods that have received more than 100 total reviews across all their listings.
from pyspark.sql import functions as f

lots_of_reviews = listings.join(reviews, listings.id == reviews.listing_id, 'inner')

lots_of_reviews\
.groupBy(listings.neighbourhood_cleansed)\
.agg(f.count(reviews.listing_id).alias('review_count'))\
.filter(f.col('review_count')> 100)\
.show()

+----------------------+------------+
|neighbourhood_cleansed|review_count|
+----------------------+------------+
|            FIRST WARD|         444|
|            FIFTH WARD|         277|
|           SECOND WARD|        5615|
|           EIGHTH WARD|        1059|
|       THIRTEENTH WARD|        1765|
|            TENTH WARD|        1527|
|       FOURTEENTH WARD|        2913|
|            NINTH WARD|        1933|
|         ELEVENTH WARD|         976|
|            THIRD WARD|        1354|
|            SIXTH WARD|        8197|
|        FIFTEENTH WARD|        1316|
|          SEVENTH WARD|         656|
|           FOURTH WARD|         991|
+----------------------+------------+



26/09/07 16:30:19 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 195224 ms exceeds timeout 120000 ms
26/09/07 16:30:19 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/07 16:30:25 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora